# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining_label"].mean()
print(f"base rate: {BASE_RATE:.3f}")

print("Distributions -- mean vs median gap signals a heavy tail:")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "word_count", "search_volume"]:
    s = df[col].dropna()
    print(f"  {col}: mean={s.mean():>9.1f}  median={s.median():>8.1f}  p90={s.quantile(0.9):>9.1f}  p99={s.quantile(0.99):>9.1f}  max={s.max():>10.1f}")

Working directory: C:\Users\Laptop\Documents\fly


base rate: 0.542
Distributions -- mean vs median gap signals a heavy tail:
  impressions_90d: mean=   5200.4  median=   731.0  p90=  12136.4  p99=  73505.8  max=  517715.0
  clicks_90d: mean=     16.1  median=     1.0  p90=     32.0  p99=    253.0  max=    4178.0
  sessions_90d: mean=     37.1  median=     7.0  p90=     88.0  p99=    451.0  max=    4345.0
  word_count: mean=   3107.8  median=  2877.0  p90=   5327.0  p99=   7292.0  max=    9546.0
  search_volume: mean=    158.9  median=    10.0  p90=    110.0  p99=   2900.0  max=   74000.0


### 1. Distributions

Every traffic-count field has a heavy right tail: `impressions_90d` mean (5,200) is more than 7x
its median (731), and `search_volume` mean (159) is nearly 16x its median (10) — a handful of
very large pages/keywords pull the average far above what a typical page looks like. This is why
the model in Weeks 5-7 uses `log1p` transforms on impressions, clicks, sessions, and AI sessions
rather than raw counts — a linear model would otherwise let a few huge outliers dominate.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# Signal 1: staleness (behind FlyRank's refresh flags)
t1 = df.groupby("freshness_tier").agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
t1["lift_vs_base"] = (t1["decline_rate"] - BASE_RATE).round(3)
print("--- Signal 1: freshness_tier ---")
print(t1.round(3))

# Signal 2: CTR vs position (behind the CTR-fix logic), volume-floored
have_pos = df[df["avg_position"] > 0].copy()
vol = have_pos[have_pos["impressions_90d"] >= 100].copy()
vol["pos_band"] = pd.cut(vol["avg_position"], [0, 3, 10, 20, 50, 1e9], labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
print("\n--- Signal 2: CTR within position band (n=%d) ---" % len(vol))
for band in ["top_3", "page_1", "striking", "page_3_5", "deep"]:
    sub = vol[vol["pos_band"] == band]
    if len(sub) < 50:
        continue
    sub = sub.copy()
    sub["ctr_q"] = pd.qcut(sub["ctr"], 4, duplicates="drop", labels=False)
    g = sub.groupby("ctr_q", observed=True).agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
    print(f"[{band}] n={len(sub):,}")
    print(g.round(3).to_string())

# Signal 3 (new): demand (search_volume) tier vs decline
has_kw = df[df["search_volume"].notna()].copy()
has_kw["sv_tier"] = pd.qcut(has_kw["search_volume"].rank(method="first"), 4, labels=["low", "mid_low", "mid_high", "high"])
t3 = has_kw.groupby("sv_tier", observed=True).agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
t3["lift_vs_base"] = (t3["decline_rate"] - BASE_RATE).round(3)
print("\n--- Signal 3: search_volume tier ---")
print(t3.round(3))

--- Signal 1: freshness_tier ---
                    n  decline_rate  lift_vs_base
freshness_tier                                   
0-30            20480         0.511        -0.031
181+              174         0.471        -0.071
31-90             175         0.589         0.047
91-180           9171         0.611         0.069

--- Signal 2: CTR within position band (n=22006) ---
[top_3] n=555
         n  decline_rate
ctr_q                   
0      142         0.866
1      138         0.877
2      142         0.824
3      133         0.429
[page_1] n=8,660
          n  decline_rate
ctr_q                    
0      2241         0.750
1      2195         0.600
2      2060         0.563
3      2164         0.505
[striking] n=5,876
          n  decline_rate
ctr_q                    
0      2953         0.679
1      1481         0.585
2      1442         0.560
[page_3_5] n=6,037
          n  decline_rate
ctr_q                    
0      3071         0.592
1      1481         0.626
2   

### 2. Signal tests — three signals, three verdicts

**Signal 1 — staleness: MIXED** (repeated from Week 4). `freshness_tier` isn't monotonic:
`91-180` days shows the highest decline rate (61.1%), but `181+` — the stalest tier — declines
*less* than base (47.1%). Not a signal to trust alone.

**Signal 2 — CTR vs. position: CONFIRMED** (repeated from Week 4). Within every volume-floored
position band, lower CTR cleanly predicts higher decline — `top_3`'s bottom CTR quartile declines
86.6% of the time vs. 42.9% for the top quartile.

**Signal 3 — demand (search_volume) tier: CONFIRMED, new this week.** Monotonic and clean: the
lowest search-volume quartile declines at 62.8% (+8.6pp vs. base), the highest at 51.0% (-3.3pp
vs. base). Higher underlying keyword demand is associated with more stable pages — intuitive
(more inherent traffic gives a page more room before a dip crosses the -20% threshold), and worth
a look for a future baseline-rule iteration.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# flag-linked test: does the "quick win" assumption hold? -- a quick-win flag
# typically assumes high-demand pages sitting just off page 1 (striking
# distance, positions 11-20) are easy wins, so they SHOULD show lower decline
# risk than the rest of the visible portfolio. Test it directly.
have_pos_kw = df[(df["avg_position"] > 0) & (df["search_volume"].notna())].copy()
have_pos_kw["quick_win_candidate"] = (
    (have_pos_kw["search_volume"] >= have_pos_kw["search_volume"].median())
    & have_pos_kw["avg_position"].between(11, 20)
).astype(int)
t4 = have_pos_kw.groupby("quick_win_candidate").agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
t4["lift_vs_base"] = (t4["decline_rate"] - BASE_RATE).round(3)
print("quick_win_candidate = high demand (>= median search_volume) AND striking distance (position 11-20)")
print(t4.round(3))

quick_win_candidate = high demand (>= median search_volume) AND striking distance (position 11-20)
                         n  decline_rate  lift_vs_base
quick_win_candidate                                   
0                    23517         0.573         0.031
1                     3559         0.576         0.034


### 3. The flag-linked test — "quick win" assumption: FALSE

A "quick win" flag typically assumes high-demand pages sitting just off page 1 (positions 11-20)
are easy, low-risk targets. Tested directly: quick-win candidates decline at 57.6% vs. 57.3% for
everyone else at that visibility level — a difference of 0.3 points, well within noise. **The
data does not support this assumption on this dataset.** This is a genuine negative result, not a
disappointing one: it's evidence *against* adding a "quick win" reason code to the Week 4/7
action rules, which would have added a plausible-sounding but empirically empty signal to the
queue.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
print("Signal verdicts:")
print("  1. staleness (freshness_tier):      MIXED    -- not monotonic, 181+ tier declines LESS than base")
print("  2. CTR vs position (volume-floored): CONFIRMED -- clean, monotonic in every position band")
print("  3. search_volume tier:               CONFIRMED -- monotonic, low volume +8.6pp vs base, high volume -3.3pp")
print("  4. 'quick win' flag assumption:       FALSE    -- candidates decline at ~the same rate as everyone else")

Signal verdicts:
  1. staleness (freshness_tier):      MIXED    -- not monotonic, 181+ tier declines LESS than base
  2. CTR vs position (volume-floored): CONFIRMED -- clean, monotonic in every position band
  3. search_volume tier:               CONFIRMED -- monotonic, low volume +8.6pp vs base, high volume -3.3pp
  4. 'quick win' flag assumption:       FALSE    -- candidates decline at ~the same rate as everyone else


### 4. What this means in practice

A content team should trust CTR-vs-position and demand tier as real, checkable signals — both are
clean and monotonic here. Staleness alone is weaker than it sounds and shouldn't anchor a rule by
itself. And "this page looks like an easy win" (high demand, close to page 1) is not, on this
data, actually associated with lower decline risk — worth remembering before that intuition gets
coded into a permanent flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.